In [54]:
import pandas as pd
import re
import json
import tqdm

### READ META INFO

In [55]:
# gene info
gene_info = pd.read_csv('../../../raw/ECO/eco_gene_info.txt', sep='\t', names=['uni_id','name','id','full_name'])
name2uniid = {}  # NOTE: use lower case
id2uniid = {}
all_gene_uniid = set(gene_info['uni_id'].unique())
for i in range(gene_info.shape[0]):
    uniid     = gene_info.iloc[i]['uni_id']
    name      = gene_info.iloc[i]['name']
    locus_tag = gene_info.iloc[i]['id']
    name2uniid[name.lower()] = uniid
    id2uniid[locus_tag] = uniid

In [56]:
# complex info
complex_info = pd.read_csv('../../../raw/ECO/eco_complex_info.txt', sep='\t', names=['id','subunits'])
sub2cpx = {}
all_complex = set()
for i in range(complex_info.shape[0]):
    cpx_id   = complex_info.iloc[i]['id']
    subunits = [x for x in complex_info.iloc[i]['subunits'].split(' ') if x in all_gene_uniid]
    if len(subunits) == 1:
        continue
    subunits = tuple(sorted(subunits))  # NOTE: tuple to ensure hashable
    sub2cpx[subunits] = cpx_id
    all_complex.add(cpx_id)

In [57]:
# reaction info
rxn_info_df = pd.read_csv('../../../raw/shared/reaction_info.txt', sep='\t')
rxnbigg = {}
for i in range(rxn_info_df.shape[0]):
    new_bigg = rxn_info_df.iloc[i]['bigg_id']
    old_bigg = rxn_info_df.iloc[i]['old_bigg_ids'].split('; ')
    old_bigg.append(new_bigg)
    for x in old_bigg:
        rxnbigg[x] = new_bigg

In [58]:
# metabolite info
met_info_df = pd.read_csv('../../../raw/shared/metabolite_info.txt', sep='\t')
metabigg = {}
for i in range(met_info_df.shape[0]):
    new_bigg = met_info_df.iloc[i]['bigg_id']
    try:
        old_bigg = met_info_df.iloc[i]['old_bigg_ids'].split('; ')
        old_bigg.append(new_bigg)
    except:
        old_bigg = [new_bigg]
    for x in old_bigg:
        metabigg[x] = new_bigg

inchikey2id = {}
for i in range(met_info_df.shape[0]):
    met_bigg_id = met_info_df.iloc[i]['bigg_id']
    if met_bigg_id.endswith('_c'):
        inchikey = met_info_df.iloc[i]['inchikey']
        if pd.isna(inchikey):
            continue
        inchikey2id['-'.join(inchikey.split('-')[:-1])] = met_bigg_id

In [ ]:
# 流通代谢物
common_meta_df = pd.read_csv('../../../raw/shared/common_metabolite.txt', sep='\t')
common_meta = list(common_meta_df['met_id_in_1515'].unique())
common_meta = [metabigg[x] if x in metabigg.keys() else x for x in common_meta]
# strip compartment suffix to match metabolites in any compartment (_m, _x, _e, etc.)
common_meta_base = {m.rsplit('_', 1)[0] for m in common_meta}

In [60]:
# GEM model (iML1515)
with open('../../../raw/ECO/iML1515.json', 'r', encoding='utf-8') as file:
    gem = json.load(file)

### GENERATE TRIPLES

In [61]:
# complex triples: Protein --Forms--> Complex
triples = []

for k, v in sub2cpx.items():
    for subp in k:
        triples.append(('ECO_Protein:' + subp, 'Forms', 'ECO_Complex:' + v))

In [ ]:
# reaction triples: Reaction --Produces/Consumes--> Metabolite (skip common metabolites, skip BIOMASS)
# reversible reactions (lower_bound < 0) are expanded with a _-1 reverse direction
for x in tqdm.tqdm(gem['reactions']):
    if 'BIOMASS' in x['id']:
        continue
    rxn_id = rxnbigg.get(x['id'], x['id'])
    reversible = x.get('lower_bound', 0) < 0

    for k, v in x['metabolites'].items():
        k = metabigg.get(k, k)
        if k in common_meta or k.rsplit('_', 1)[0] in common_meta_base:
            continue
        if v > 0:
            triples.append(('Reaction:' + rxn_id, 'Produces', 'Metabolite:' + k))
            if reversible:
                triples.append(('Reaction:' + rxn_id + '_-1', 'Consumes', 'Metabolite:' + k))
        elif v < 0:
            triples.append(('Reaction:' + rxn_id, 'Consumes', 'Metabolite:' + k))
            if reversible:
                triples.append(('Reaction:' + rxn_id + '_-1', 'Produces', 'Metabolite:' + k))

100%|██████████| 2712/2712 [00:00<00:00, 28262.02it/s]


In [ ]:
# catalyze triples: Protein --Catalyzes--> Reaction
# 'and' rules are expanded to individual proteins (not collapsed to Complex)
# reversible reactions also get a _-1 entry
# skip BIOMASS and reactions where all metabolites are common (no specific metabolites)
for x in tqdm.tqdm(gem['reactions']):
    if 'BIOMASS' in x['id']:
        continue
    rxn_id = rxnbigg.get(x['id'], x['id'])
    reversible = x.get('lower_bound', 0) < 0

    if x['gene_reaction_rule'] == '':
        continue
    mets_mapped = [metabigg.get(k, k) for k in x['metabolites']]
    if all(m in common_meta or m.rsplit('_', 1)[0] in common_meta_base for m in mets_mapped):
        continue

    for rule_token in x['gene_reaction_rule'].split(' or '):
        if 'and' in rule_token:
            locus_tags = re.findall(r'b.{4}', rule_token)
            proteins = [id2uniid[lt] for lt in locus_tags if lt in id2uniid]
        else:
            try:
                proteins = [id2uniid[rule_token]]
            except:
                continue
        for prot in proteins:
            triples.append(('ECO_Protein:' + prot, 'Catalyzes', 'Reaction:' + rxn_id))
            if reversible:
                triples.append(('ECO_Protein:' + prot, 'Catalyzes', 'Reaction:' + rxn_id + '_-1'))

  0%|          | 0/2712 [00:00<?, ?it/s]

100%|██████████| 2712/2712 [00:00<00:00, 49430.10it/s]


In [64]:
# ppi triples: Protein --PPI--> Protein  (STRING DB, combined_score >= 800)
ppi_raw_df = pd.read_csv('../../../raw/ECO/511145.protein.links.full.v12.0.txt', sep=' ')
ppi_df = ppi_raw_df[ppi_raw_df['combined_score'] >= 800]
for i in tqdm.tqdm(range(ppi_df.shape[0])):
    try:
        src_uni = id2uniid[ppi_df.iloc[i]['protein1'].split('.')[1]]
        dst_uni = id2uniid[ppi_df.iloc[i]['protein2'].split('.')[1]]
    except KeyError:
        continue
    triples.append(('ECO_Protein:' + src_uni, 'PPI', 'ECO_Protein:' + dst_uni))

100%|██████████| 48828/48828 [00:09<00:00, 5272.77it/s]


In [65]:
# complex GO triples
cpx_go_df = pd.read_csv('../../../raw/ECO/QuickGO-annotations-complex.tsv', sep='\t')
cpx_go_df = cpx_go_df[cpx_go_df['GENE PRODUCT ID'].isin(all_complex)]
cpx_go_df = cpx_go_df[cpx_go_df['QUALIFIER'].isin(['enables', 'involved_in', 'part_of'])]
for i in range(cpx_go_df.shape[0]):
    entity_id = cpx_go_df.iloc[i]['GENE PRODUCT ID']
    qualifier  = cpx_go_df.iloc[i]['QUALIFIER'].capitalize()
    go_term    = cpx_go_df.iloc[i]['GO TERM']
    triples.append(('ECO_Complex:' + entity_id, qualifier, go_term))

In [66]:
# protein GO triples
prot_go_df = pd.read_csv('../../../raw/ECO/QuickGO-annotations-protein.tsv', sep='\t')
prot_go_df = prot_go_df[prot_go_df['GENE PRODUCT ID'].isin(all_gene_uniid)]
prot_go_df = prot_go_df[prot_go_df['QUALIFIER'].isin(['enables', 'involved_in', 'part_of', 'acts_upstream_of_or_within', 'is_active_in'])]
for i in range(prot_go_df.shape[0]):
    entity_id = prot_go_df.iloc[i]['GENE PRODUCT ID']
    qualifier  = prot_go_df.iloc[i]['QUALIFIER'].capitalize()
    go_term    = prot_go_df.iloc[i]['GO TERM']
    triples.append(('ECO_Protein:' + entity_id, qualifier, go_term))

In [67]:
# protein KO triples (KEGG)
ko_df = pd.read_csv('../../../raw/shared/genes_ko.list', sep='\t', names=['gene', 'ko'])
ko_df = ko_df[ko_df['gene'].str.startswith('eco:')]
for i in range(ko_df.shape[0]):
    try:
        prot_uni = id2uniid[ko_df.iloc[i]['gene'].split(':')[1]]
    except:
        continue
    ko_id = 'KO:' + ko_df.iloc[i]['ko'].split(':')[1]
    triples.append(('ECO_Protein:' + prot_uni, 'Has_KO_Item', ko_id))

In [68]:
# common_triples: Metabolite --Belongs_to--> Pathway, Pathway --in--> Pathway, etc.
# Filter: drop triples where any Metabolite: node is not in the original KG

# Collect all metabolite IDs present in existing KG triples
kg_mets = set()
for h, r, t in triples:
    if h.startswith('Metabolite:'):
        kg_mets.add(h)
    if t.startswith('Metabolite:'):
        kg_mets.add(t)

# Load and filter common_triples.txt
common_triples_df = pd.read_csv('../../../raw/shared/common_triples.txt', sep='\t', names=['from', 'rel', 'to'])

def _met_in_kg(row):
    for val in (row['from'], row['to']):
        if val.startswith('Metabolite:') and val not in kg_mets:
            return False
    return True

common_triples_filtered = common_triples_df[common_triples_df.apply(_met_in_kg, axis=1)]
print(f"common_triples: {len(common_triples_df)} total -> {len(common_triples_filtered)} after filtering out unknown mets")

for _, row in common_triples_filtered.iterrows():
    triples.append((row['from'], row['rel'], row['to']))

common_triples: 2919 total -> 2087 after filtering out unknown mets


In [69]:
### OUTPUT kg.txt

In [70]:
kg = pd.DataFrame(triples, columns=['from', 'rel', 'to'])
kg = kg.drop_duplicates()
print(kg.shape)
kg.head()

(99011, 3)


,from,rel,to
0,ECO_Protein:P76052,Forms,ECO_Complex:CPX-28
1,ECO_Protein:P77357,Forms,ECO_Complex:CPX-28
2,ECO_Protein:P23873,Forms,ECO_Complex:CPX-180
3,ECO_Protein:P23874,Forms,ECO_Complex:CPX-180
4,ECO_Protein:P0AFC3,Forms,ECO_Complex:CPX-243


In [71]:
kg.to_csv('../../../processed/kg/ECO/kg_reproduced.txt', sep='\t', index=False, header=False)